# Full Pipeline Test — Engineering Diagnostic Reasoning Engine
Standalone notebook: loads data fresh, no dependency on other notebooks.

In [1]:
import sys, os

# find project root (folder containing 'apps') regardless of where this notebook runs from
current = os.getcwd()
while not os.path.exists(os.path.join(current, 'apps')) and current != os.path.dirname(current):
    current = os.path.dirname(current)
sys.path.append(current)

print('Project root found:', current)

Project root found: c:\Users\hp\predictive-maintenance-diagnosis


In [2]:
import pandas as pd
import numpy as np

# load the processed dataset (adjust filename if yours differs)
data_path = os.path.join(current, 'data', 'processed', 'mafaulda_features_final.csv')
df = pd.read_csv(data_path)

print(df.shape)
print(df['fault_type'].value_counts())

(244000, 74)
fault_type
underhang_bearing          68076
overhang_bearing           62586
imbalance                  40626
vertical_misalignment      36722
horizontal_misalignment    24034
normal                     11956
Name: count, dtype: int64


In [3]:
# define feature columns — exclude label and file_id
feature_cols = [c for c in df.columns if c not in ['fault_type', 'file_id']]
print(len(feature_cols), 'features')
print(feature_cols[:5], '...')

72 features
['tachometer_max', 'tachometer_min', 'tachometer_mean', 'tachometer_sd', 'tachometer_rms'] ...


In [4]:
# Recreate a proper file-level split (matches how the model was actually trained/evaluated)
# If you don't have file_id in this CSV, this cell falls back to a simple random split instead.

if 'file_id' in df.columns:
    from sklearn.model_selection import GroupShuffleSplit

    gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df['file_id']))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
else:
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['fault_type'], random_state=42)
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df['fault_type']
y_test = test_df['fault_type']

print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (195200, 72) Test: (48800, 72)


In [5]:
from apps.models.predictor import FaultPredictor

predictor = FaultPredictor()
print('Model, scaler, label encoder loaded successfully')
print('Classes:', predictor.label_encoder.classes_)

Model, scaler, label encoder loaded successfully
Classes: ['horizontal_misalignment' 'imbalance' 'normal' 'overhang_bearing'
 'underhang_bearing' 'vertical_misalignment']


In [6]:
# Build a SCALED background sample for SHAP (the explainer needs scaled data,
# same scale the model actually operates on)
background_raw = X_train.sample(100, random_state=42)
background = predictor.scaler.transform(background_raw)

print('Background shape:', background.shape)

Background shape: (100, 72)


In [7]:
from apps.pipeline.diagnostic_pipeline import DiagnosticEngine

engine = DiagnosticEngine(background_data=background, feature_cols=feature_cols)
print('Diagnostic engine ready')

Diagnostic engine ready


In [8]:
import json

# pick one test sample and run the full 6-step diagnosis
sample_idx = 0
sample_features = X_test.iloc[sample_idx].to_dict()
true_label = y_test.iloc[sample_idx]

result = engine.diagnose(sample_features)

print('True label:', true_label)
print(json.dumps(result, indent=2, default=str))

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


True label: normal
{
  "predicted_class": "normal",
  "confidence": 0.9635307788848877,
  "top_features": [
    {
      "feature": "overhang_axial_max",
      "shap_value": 3.2802561649731548,
      "direction": "increases",
      "channel_category": "vibration_overhang",
      "stat_category": "peak_magnitude",
      "meaning": "Peak signal value \u2014 sensor: overhang_axial"
    },
    {
      "feature": "overhang_tangential_sd",
      "shap_value": 2.4083877622271546,
      "direction": "increases",
      "channel_category": "vibration_overhang",
      "stat_category": "general_roughness",
      "meaning": "Signal spread/roughness \u2014 sensor: overhang_tangential"
    },
    {
      "feature": "overhang_axial_sd",
      "shap_value": 1.9204631009383875,
      "direction": "increases",
      "channel_category": "vibration_overhang",
      "stat_category": "general_roughness",
      "meaning": "Signal spread/roughness \u2014 sensor: overhang_axial"
    },
    {
      "feature": "un

In [9]:
# same sample, formatted as a readable technician report
report = engine.diagnose(sample_features, as_report=True)
print(report)

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


MACHINE DIAGNOSTIC REPORT

Predicted condition: normal
Confidence: 96.4%

--- Key contributing factors ---
  • overhang_axial_max (Peak signal value — sensor: overhang_axial) — increases confidence [+2.543]
  • overhang_tangential_sd (Signal spread/roughness — sensor: overhang_tangential) — increases confidence [+2.489]
  • underhang_tangential_skewness (Signal asymmetry — sensor: underhang_tangential) — increases confidence [+1.889]
  • underhang_axial_form (Waveform shape descriptor — sensor: underhang_axial) — decreases confidence [-1.751]
  • underhang_tangential_sd (Signal spread/roughness — sensor: underhang_tangential) — increases confidence [+1.709]

--- Diagnosed phenomenon ---
  Healthy Operation

--- Probable root cause(s) ---
  • No fault detected (likelihood: high)
    → Recommended action: No action needed. Continue routine monitoring.



In [10]:
# run on a few more samples across different classes, to sanity-check broadly
for idx in [0, 100, 500, 1000, 2000]:
    if idx >= len(X_test):
        continue
    feats = X_test.iloc[idx].to_dict()
    true = y_test.iloc[idx]
    res = engine.diagnose(feats)
    print(f"[{idx}] True: {true} | Predicted: {res['predicted_class']} ({res['confidence']:.1%}) | Phenomenon: {res['phenomenon']['phenomenon_id']}")

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[0] True: normal | Predicted: normal (96.4%) | Phenomenon: healthy_operation


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[100] True: normal | Predicted: normal (51.0%) | Phenomenon: healthy_operation


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[500] True: normal | Predicted: horizontal_misalignment (82.5%) | Phenomenon: shaft_misalignment


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[1000] True: normal | Predicted: normal (99.7%) | Phenomenon: healthy_operation


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[2000] True: normal | Predicted: normal (84.6%) | Phenomenon: healthy_operation


In [11]:
sample_indices = np.random.choice(len(X_test), size=8, replace=False)
for idx in sample_indices:
    feats = X_test.iloc[idx].to_dict()
    true = y_test.iloc[idx]
    res = engine.diagnose(feats)
    print(f"[{idx}] True: {true} | Predicted: {res['predicted_class']} ({res['confidence']:.1%}) | Phenomenon: {res['phenomenon']['phenomenon_id']}")

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[34224] True: underhang_bearing | Predicted: underhang_bearing (100.0%) | Phenomenon: impact_damage_confirmed


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[19146] True: vertical_misalignment | Predicted: vertical_misalignment (99.5%) | Phenomenon: shaft_misalignment


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


...

[29201] True: underhang_bearing | Predicted: overhang_bearing (59.1%) | Phenomenon: wear_or_lubrication


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[45929] True: overhang_bearing | Predicted: overhang_bearing (100.0%) | Phenomenon: wear_or_lubrication


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[37367] True: underhang_bearing | Predicted: underhang_bearing (100.0%) | Phenomenon: wear_or_lubrication


In [12]:
sample_indices = np.random.choice(len(X_test), size=20, replace=False)
seen_phenomena = set()

for idx in sample_indices:
    feats = X_test.iloc[idx].to_dict()
    true = y_test.iloc[idx]
    res = engine.diagnose(feats)
    seen_phenomena.add(res['phenomenon']['phenomenon_id'])
    print(f"[{idx}] True: {true} | Predicted: {res['predicted_class']} ({res['confidence']:.1%}) | Phenomenon: {res['phenomenon']['phenomenon_id']}")

print("\nUnique phenomena observed:", seen_phenomena)

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[33269] True: underhang_bearing | Predicted: underhang_bearing (100.0%) | Phenomenon: impact_damage_confirmed


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[41159] True: overhang_bearing | Predicted: overhang_bearing (100.0%) | Phenomenon: wear_or_lubrication


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


...

[22747] True: vertical_misalignment | Predicted: vertical_misalignment (99.9%) | Phenomenon: shaft_misalignment


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[23510] True: vertical_misalignment | Predicted: vertical_misalignment (99.9%) | Phenomenon: shaft_misalignment


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[4816] True: horizontal_misalignment | Predicted: horizontal_misalignment (99.8%) | Phenomenon: shaft_misalignment

Unique phenomena observed: {'shaft_misalignment', 'mass_imbalance', 'wear_or_lubrication', 'impact_damage_confirmed'}


In [13]:
for idx in sample_indices:
    feats = X_test.iloc[idx].to_dict()
    true = y_test.iloc[idx]
    res = engine.diagnose(feats)
    if res['phenomenon']['phenomenon_id'] == 'unmatched_pattern':
        print(f"Index: {idx}")
        print(f"True label: {true}")
        print(f"Predicted: {res['predicted_class']} ({res['confidence']:.1%})")
        print("\nTop features:")
        for f in res['top_features']:
            print(f"  {f['feature']} -> channel: {f['channel_category']}, stat: {f['stat_category']}")

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted w